<a href="https://colab.research.google.com/github/debanjanhati2-boop/Basic-Chatbot/blob/main/Automated_Data_Cleaning_%26_Schema_Transformation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
pip install pandas numpy

In [29]:
import pandas as pd
import numpy as np
import re
import json
import logging
from pathlib import Path

# Create folders

In [30]:
Path("output").mkdir(exist_ok=True)
Path("logs").mkdir(exist_ok=True)

# Configure logging

In [31]:
logging.basicConfig(
    filename="logs/cleaning.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# Load dirty dataset

In [32]:
df = pd.read_csv("dirty_sales_data.csv", dtype="string")
initial_rows = len(df)
print("Original dataset:")
print(df.head())

Original dataset:
   order_id customer_name         product  order_date quantity unit_price  \
0  ORD-0001    Kabir Khan  Wireless Mouse  23-10-2026        4      15017   
1  ORD-0002      Mira Das          Webcam  18-04-2026        5   19119.47   
2  ORD-0003    Ananya Roy        Keyboard  23-04-2026        1     7894.6   
3  ORD-0004    Kabir Khan         Monitor  24-06-2026        2   23243.16   
4  ORD-0005      Mira Das  Wireless Mouse  23-08-2026        2   15887.24   

  total_amount     city     status  
0     60069.76  Kolkata  Delivered  
1        95597  Chennai  Delivered  
2         7895    Delhi  Delivered  
3     46486.32  Unknown  Delivered  
4     31774.48     Pune    Pending  


#Handle missing values

In [33]:
def clean_missing_values(data):
    for col in ["customer_name", "product", "city", "status"]:
        data[col] = data[col].fillna("Unknown")
    data["quantity"] = pd.to_numeric(
        data["quantity"], errors="coerce"
    ).fillna(1).astype("Int64")
    logging.info("Missing values handled")
    return data

#Normalize currency

In [34]:
def normalize_currency(value):
    if pd.isna(value):
        return np.nan
    value = str(value)
    value = re.sub(r"(?i)(INR|RS\.?|RUPEES?)", "", value)
    value = value.replace("₹", "").replace(",", "").strip()
    value = re.sub(r"[^\d.\-]", "", value)
    try:
        return float(value)
    except ValueError:
        return np.nan

#Normalize dates

In [35]:
def normalize_dates(data):
    data["order_date"] = pd.to_datetime(
        data["order_date"],
        errors="coerce",
        dayfirst=True,
        format="mixed"
    )
    return data

#Standardize text

In [36]:
def standardize_text(data):
    columns = ["customer_name", "product", "city", "status"]

    for col in columns:
        data[col] = (
            data[col].astype("string")
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
            .str.title()
        )
        data["order_id"] = data["order_id"].str.strip().str.upper()
    return data


#Remove duplicates

In [37]:
def remove_duplicates(data):
    before = len(data)

    data.drop_duplicates(inplace=True)
    data.drop_duplicates(
        subset=["order_id"],
        keep="first",
        inplace=True
    )
    removed = before - len(data)
    logging.info("Duplicates removed: %s", removed)
    return data, removed

#Validate schema

In [38]:
def validate_schema(data):
    required_columns = [
        "order_id", "customer_name", "product",
        "order_date", "quantity", "unit_price",
        "total_amount", "city", "status"
    ]
    errors = []

    missing = set(required_columns) - set(data.columns)

    if missing:
        errors.append(f"Missing columns: {sorted(missing)}")
        return errors
    if data["order_id"].isna().any():
        errors.append("Null order IDs found")

    if data["order_id"].duplicated().any():
        errors.append("Duplicate order IDs found")
    if not pd.api.types.is_datetime64_any_dtype(data["order_date"]):
        errors.append("Invalid date data type")
    for col in ["quantity", "unit_price", "total_amount"]:
        if not pd.api.types.is_numeric_dtype(data[col]):
            errors.append(f"{col} must be numeric")
    if (data["quantity"].dropna() <= 0).any():
      errors.append("Quantity must be positive")
    if (data["unit_price"].dropna() < 0).any():
        errors.append("Unit price cannot be negative")
    if (data["total_amount"].dropna() < 0).any():
        errors.append("Total amount cannot be negative")
    return errors

#MAIN CLEANING PROCESS

In [39]:
# Handle missing values
df = clean_missing_values(df)

# Normalize currency
for col in ["unit_price", "total_amount"]:
    df[col] = pd.to_numeric(
        df[col].apply(normalize_currency),
        errors="coerce"
    )

    # Normalize dates
df = normalize_dates(df)

# Standardize text
df = standardize_text(df)

# Recalculate missing totals where possible

In [40]:
mask = (
    df["total_amount"].isna()
    & df["unit_price"].notna()
)

df.loc[mask, "total_amount"] = (
    df.loc[mask, "unit_price"] * df.loc[mask, "quantity"]
).round(2)

# Remove duplicates

In [41]:
df, duplicates_removed = remove_duplicates(df)

# Enforce column order

In [42]:
required_columns = [
    "order_id", "customer_name", "product",
    "order_date", "quantity", "unit_price",
    "total_amount", "city", "status"
]
df = df[required_columns]

# Validate

In [43]:
errors = validate_schema(df)

# Export CSV

In [44]:
output = df.copy()
output["order_date"] = output["order_date"].dt.strftime("%Y-%m-%d")
output.to_csv(
    "output/cleaned_sales_data.csv",
    index=False,
    encoding="utf-8-sig"
)

# Export JSON

In [45]:
output.to_json(
    "output/cleaned_sales_data.json",
    orient="records",
    indent=2,
    force_ascii=False
)

# Generate validation report

In [46]:
report = {
    "input_records": initial_rows,
    "cleaned_records": len(df),
    "duplicates_removed": duplicates_removed,
    "missing_values_after_cleaning": {
        col: int(df[col].isna().sum())
        for col in df.columns
    },
    "schema_validation": "PASSED" if not errors else "FAILED",
    "validation_errors": errors,
    "columns": list(df.columns),
    "data_types": {
        col: str(dtype)
        for col, dtype in df.dtypes.items()
    }
}
with open("logs/quality_summary.json", "w") as file:
    json.dump(report, file, indent=4)
with open("logs/validation_report.txt", "w") as file:
    file.write("AUTOMATED DATA CLEANING REPORT\n")
    file.write("=" * 40 + "\n")
    for key, value in report.items():
        file.write(f"{key}: {value}\n")
if errors:
    print("SCHEMA VALIDATION FAILED")
    for error in errors:
        print("-", error)
else:
    print("\nSCHEMA VALIDATION PASSED")

print("\nData cleaning completed successfully!")
print("Original records:", initial_rows)
print("Cleaned records:", len(df))
print("Duplicates removed:", duplicates_removed)
print("CSV and JSON files exported.")



SCHEMA VALIDATION PASSED

Data cleaning completed successfully!
Original records: 130
Cleaned records: 130
Duplicates removed: 0
CSV and JSON files exported.
